# [3장 3강] - Attention Weight와 Context Vector 해석 (1)

<aside>
🎯

**실습 목표**

- Attention Weight와 Value가 Context Vector에 기여하는 방식을 계산합니다.
- Entropy를 이용해 attention 분포가 집중되었는지 확인합니다.
- 여러 head의 관계를 비교해 공통으로 강한 연결을 찾습니다.
</aside>

---

## 핵심 실습. Context Vector 기여도 분해

### 시작 코드

```python
import torch

weights = torch.tensor([0.6, 0.3, 0.1])
V = torch.tensor([
    [10.0, 0.0],
    [0.0, 10.0],
    [5.0, 5.0],
])
tokens = ["모델", "보안", "배포"]
```

### 수행해야 할 작업

1. 각 token의 `weight * value`를 계산하는 `decompose_context` 함수를 작성하세요.
2. 기여 벡터의 합으로 context vector를 만드세요.
3. token별 weight와 기여 벡터를 리포트로 반환하세요.
4. 수동 가중합과 행렬곱 결과가 같은지 검증하세요.

  **해설**
    
  Weight가 큰 token이 항상 모든 차원에서 가장 큰 기여를 만드는 것은 아닙니다. 실제 기여는 weight와 Value 벡터 값이 함께 결정합니다.

In [1]:
import torch

weights = torch.tensor([0.6, 0.3, 0.1])
V = torch.tensor([[10., 0.], [0., 10.], [5., 5.]])
tokens = ["모델", "보안", "배포"]


def decompose_context(weights, V, tokens):
    if weights.ndim != 1 or weights.numel() != V.size(0):
        raise ValueError("weight 수와 Value의 token 수가 다릅니다.")

    # 각 Value 행에 해당 token의 weight를 곱합니다.
    contributions = weights.unsqueeze(-1) * V
    context = contributions.sum(dim=0)

    report = [
        {
            "token": token,
            "weight": float(weight),
            "contribution": contribution.tolist(),
        }
        for token, weight, contribution in zip(tokens, weights, contributions)
    ]
    return context, report


context, report = decompose_context(weights, V, tokens)
print("context:", context)
print(report)

assert torch.allclose(context, weights @ V)
assert torch.allclose(context, torch.tensor([6.5, 3.5]))

context: tensor([6.5000, 3.5000])
[{'token': '모델', 'weight': 0.6000000238418579, 'contribution': [6.0, 0.0]}, {'token': '보안', 'weight': 0.30000001192092896, 'contribution': [0.0, 3.0]}, {'token': '배포', 'weight': 0.10000000149011612, 'contribution': [0.5, 0.5]}]


## 핵심 보조 실습. Attention 집중도 계산

### 시작 코드

```python
attention_rows = torch.tensor([
    [0.90, 0.05, 0.05],
    [0.34, 0.33, 0.33],
    [0.60, 0.30, 0.10],
])
```

### 수행해야 할 작업

1. 행별 Shannon entropy를 계산하는 `attention_entropy` 함수를 작성하세요.
2. `-sum(p * log(p))`를 사용하되 0에 안전하게 처리하세요.
3. 가장 집중된 행과 가장 고르게 분산된 행을 찾으세요.
4. 최대 entropy `log(token_count)`로 나눈 normalized entropy도 반환하세요.
    
   **해설**
    
  Entropy가 낮으면 소수 위치에 weight가 집중되고, 높으면 여러 위치에 고르게 퍼집니다. 집중도가 높다고 항상 좋은 attention은 아니며 task와 layer 역할에 따라 해석해야 합니다.

In [ ]:
import math
import torch

attention_rows = torch.tensor([
    [0.90, 0.05, 0.05],
    [0.34, 0.33, 0.33],
    [0.60, 0.30, 0.10],
])


def attention_entropy(weights, eps=1e-12):
    if not torch.allclose(weights.sum(dim=-1), torch.ones(weights.size(0)), atol=1e-6):
        raise ValueError("각 행의 확률 합이 1이어야 합니다.")

    safe_weights = weights.clamp_min(eps)
    entropy = -(safe_weights * safe_weights.log()).sum(dim=-1)
    normalized = entropy / math.log(weights.size(-1))
    return entropy, normalized


entropy, normalized = attention_entropy(attention_rows)
print("entropy:", entropy)
print("normalized:", normalized)
print("가장 집중된 행:", int(entropy.argmin()))
print("가장 분산된 행:", int(entropy.argmax()))

assert entropy.argmin().item() == 0
assert entropy.argmax().item() == 1

## 참고·심화 실습. 여러 Head의 공통 관계 찾기

### 시작 코드

```python
head_weights = torch.tensor([
    [[0.7, 0.2, 0.1], [0.2, 0.7, 0.1], [0.3, 0.2, 0.5]],
    [[0.6, 0.3, 0.1], [0.1, 0.8, 0.1], [0.4, 0.1, 0.5]],
])  # [H, T, T]
tokens = ["private", "llm", "serving"]
```

### 수행해야 할 작업

1. Head 평균 attention을 계산하세요.
2. Query별 평균 weight가 가장 큰 Key를 찾으세요.
3. 각 head에서 같은 Key가 1위인지 확인해 `unanimous`를 표시하세요.
4. 결과를 token 이름으로 반환하세요.
   
   **해설**
    
  Head 평균은 전체 경향을 빠르게 보는 방법이지만, 서로 다른 역할을 가진 head를 평균내면 중요한 차이가 사라질 수 있습니다. 평균과 head별 결과를 함께 보세요.

In [ ]:
import torch

head_weights = torch.tensor([
    [[0.7, 0.2, 0.1], [0.2, 0.7, 0.1], [0.3, 0.2, 0.5]],
    [[0.6, 0.3, 0.1], [0.1, 0.8, 0.1], [0.4, 0.1, 0.5]],
])
tokens = ["private", "llm", "serving"]


def summarize_heads(head_weights, tokens):
    if head_weights.ndim != 3:
        raise ValueError("head_weights는 [H, T, T]여야 합니다.")

    mean_weights = head_weights.mean(dim=0)
    per_head_top = head_weights.argmax(dim=-1)  # [H, T]
    mean_top = mean_weights.argmax(dim=-1)

    report = []
    for query_index, key_index in enumerate(mean_top.tolist()):
        unanimous = bool((per_head_top[:, query_index] == key_index).all())
        report.append({
            "query": tokens[query_index],
            "top_key": tokens[key_index],
            "mean_weight": round(float(mean_weights[query_index, key_index]), 4),
            "unanimous": unanimous,
        })
    return mean_weights, report


mean_weights, report = summarize_heads(head_weights, tokens)
print(mean_weights)
print(report)
assert all(item["unanimous"] for item in report)